# 04 - LangChain Workflow Completo: Detección de Anomalías

Este notebook demuestra el workflow completo de LangChain para análisis de transferencias usando:
- Pydantic para validación
- LangChain para orquestación
- Gemini 2.5 Flash con salida estructurada
- Manejo de errores

## PASO 1: Importar Pydantic y LangChain

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Agregar src al path
sys.path.insert(0, '../src')

# Importar Pydantic para validación
from pydantic import BaseModel, Field

# Importar LangChain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# Cargar API key
load_dotenv(dotenv_path="../.env")
api_key = os.getenv("GOOGLE_API_KEY")

print("Importaciones completadas:")
print("✓ Pydantic (BaseModel, Field)")
print("✓ LangChain (ChatGoogleGenerativeAI, PromptTemplate)")
print(f"✓ API Key cargada: {'Si' if api_key else 'No'}")

## PASO 2: Definir Modelos Pydantic

In [ ]:
# Modelo de ENTRADA - Valida datos antes de enviar al LLM
class TransferenceInput(BaseModel):
    """Entrada validada por Pydantic"""
    id_movimiento: str = Field(
        description="ID numerico de 8 digitos",
        pattern="^\\d{8}$"
    )
    monto: float = Field(
        description="Monto en USD (positivo)",
        gt=0
    )
    concepto: str = Field(
        description="Descripcion 1-125 caracteres",
        min_length=1,
        max_length=125
    )


# Modelo de SALIDA - Valida datos retornados por el LLM
class TransferenceAnalysis(BaseModel):
    """Salida estructurada del LLM"""
    id_movimiento: str = Field(description="ID de la transferencia")
    resultado: str = Field(
        description="Clasificacion: Usual o Inusual",
        pattern="^(Usual|Inusual)$"
    )
    razon_si_inusual: str | None = Field(
        description="Explicacion si es inusual",
        default=None
    )

print("Modelos Pydantic definidos:")
print("✓ TransferenceInput: valida entrada")
print("✓ TransferenceAnalysis: valida salida del LLM")

# Ejemplo de validacion
try:
    entrada_valida = TransferenceInput(
        id_movimiento="12345678",
        monto=1000.0,
        concepto="Test"
    )
    print(f"\n✓ Entrada valida: {entrada_valida.id_movimiento}")
except Exception as e:
    print(f"✗ Error: {e}")

# Ejemplo de validacion fallida
try:
    entrada_invalida = TransferenceInput(
        id_movimiento="XXXX",  # No son 8 digitos
        monto=1000.0,
        concepto="Test"
    )
except Exception as e:
    print(f"✓ Pydantic rechaza entrada invalida: {type(e).__name__}")

## PASO 3: Crear LLM Estructurado

In [ ]:
# Crear instancia del modelo LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.3,  # Bajo = respuestas consistentes
)

print(f"LLM creado: {llm.model}")
print(f"Temperature: {llm.temperature}")

# CLAVE: Crear LLM estructurado
# with_structured_output() obliga al modelo a retornar datos que validen contra TransferenceAnalysis
structured_llm = llm.with_structured_output(TransferenceAnalysis)

print("\n✓ LLM estructurado creado con with_structured_output(TransferenceAnalysis)")
print("  Esto asegura que el modelo retorne datos validos segun el modelo Pydantic")

## PASO 4: Crear Prompt Template Descriptor

In [ ]:
# Crear el prompt que describe la tarea usando PromptTemplate
prompt = PromptTemplate.from_template("""Eres un experto en deteccion de fraude financiero.
Analiza transferencias y clasifica como 'Usual' o 'Inusual'.

Banderas rojas: offshore, anonimo, lavado de dinero, paraiso fiscal.
Normales: alquiler, servicios, compras, pagos regulares.

Analiza:
- ID: {id_movimiento}
- Monto: ${monto:,.2f} USD
- Concepto: "{concepto}"

Retorna analisis estructurado. Se conciso y preciso.""")

print("Prompt template creado con PromptTemplate:")
print("✓ Variables: {id_movimiento}, {monto}, {concepto}")

# Ver como se ve el prompt renderizado
test_input = {
    "id_movimiento": "12345601",
    "monto": 1500.50,
    "concepto": "Pago de servicios"
}

prompt_text = prompt.format(**test_input)
print("\nPrompt renderizado:")
print(prompt_text[:150] + "...")

## PASO 5: Implementar Manejo de Errores (try/except)

In [ ]:
def analyze_transfer(transfer: TransferenceInput) -> TransferenceAnalysis | None:
    """Analizar transferencia con manejo de errores."""
    try:
        # 1. Validar entrada (Pydantic)
        if not isinstance(transfer, TransferenceInput):
            raise TypeError("Debe ser TransferenceInput")
        
        # 2. Construir prompt usando PromptTemplate.format()
        prompt_text = prompt.format(
            id_movimiento=transfer.id_movimiento,
            monto=transfer.monto,
            concepto=transfer.concepto
        )
        
        # 3. Invocar modelo (retorna TransferenceAnalysis validado)
        analysis = structured_llm.invoke(prompt_text)
        
        # 4. Pydantic valida automaticamente la salida
        if not isinstance(analysis, TransferenceAnalysis):
            raise ValueError("Salida no es TransferenceAnalysis valido")
        
        return analysis
    
    except TypeError as e:
        print(f"[ERROR] Tipo invalido: {e}")
        return None
    
    except ValueError as e:
        print(f"[ERROR] Validacion: {e}")
        return None
    
    except Exception as e:
        print(f"[ERROR] Inesperado: {e}")
        return None

print("Funcion analyze_transfer creada con manejo de errores:")
print("✓ try: invocar modelo")
print("✓ except TypeError: tipo invalido")
print("✓ except ValueError: validacion Pydantic")
print("✓ except Exception: errores inesperados")

## PASO 6: Probar con Ejemplo 1 (Transferencia USUAL)

In [ ]:
print("ANALISIS 1: Transferencia USUAL")
print("=" * 70)

transfer_usual = TransferenceInput(
    id_movimiento="12345601",
    monto=1500.00,
    concepto="Pago de servicios mensuales a proveedor"
)

print(f"Entrada:")
print(f"  ID: {transfer_usual.id_movimiento}")
print(f"  Monto: ${transfer_usual.monto:,.2f}")
print(f"  Concepto: {transfer_usual.concepto}")
print()

result = analyze_transfer(transfer_usual)

if result:
    print(f"Salida (TransferenceAnalysis):")
    print(f"  ID: {result.id_movimiento}")
    print(f"  Resultado: {result.resultado}")
    if result.razon_si_inusual:
        print(f"  Razon: {result.razon_si_inusual}")
    else:
        print(f"  Razon: N/A (es usual)")

## PASO 7: Probar con Ejemplo 2 (Transferencia INUSUAL)

In [ ]:
print("ANALISIS 2: Transferencia INUSUAL")
print("=" * 70)

transfer_inusual = TransferenceInput(
    id_movimiento="12345602",
    monto=50000.00,
    concepto="Transferencia anonima a cuenta offshore sin documentacion"
)

print(f"Entrada:")
print(f"  ID: {transfer_inusual.id_movimiento}")
print(f"  Monto: ${transfer_inusual.monto:,.2f}")
print(f"  Concepto: {transfer_inusual.concepto}")
print()

result = analyze_transfer(transfer_inusual)

if result:
    print(f"Salida (TransferenceAnalysis):")
    print(f"  ID: {result.id_movimiento}")
    print(f"  Resultado: {result.resultado}")
    if result.razon_si_inusual:
        print(f"  Razon: {result.razon_si_inusual}")

## RESUMEN DEL WORKFLOW COMPLETO

In [ ]:
print("WORKFLOW COMPLETO DE LANGCHAIN:")
print("=" * 70)
print()
print("1. IMPORTAR:")
print("   - Pydantic (BaseModel, Field) para validacion")
print("   - LangChain (ChatGoogleGenerativeAI, PromptTemplate)")
print()
print("2. DEFINIR MODELOS:")
print("   - TransferenceInput: valida ENTRADA (id, monto, concepto)")
print("   - TransferenceAnalysis: valida SALIDA (resultado, razon)")
print()
print("3. CREAR LLM ESTRUCTURADO:")
print("   - llm = ChatGoogleGenerativeAI(...)")
print("   - structured_llm = llm.with_structured_output(TransferenceAnalysis)")
print()
print("4. CREAR PROMPT:")
print("   - PromptTemplate.from_template(...)")
print("   - Con variables {id_movimiento}, {monto}, {concepto}")
print()
print("5. IMPLEMENTAR ERRORES:")
print("   - try: invocar structured_llm.invoke(prompt_text)")
print("   - except: capturar TypeError, ValueError, Exception")
print()
print("6. PROBAR:")
print("   - Crear TransferenceInput validada por Pydantic")
print("   - Invocar analyze_transfer()")
print("   - Recibir TransferenceAnalysis validada")
print()
print("FLUJO DATOS:")
print("   TransferenceInput --validate--> prompt --format--> LLM --validate--> TransferenceAnalysis")
print("   (Pydantic)                 (PromptTemplate)   (Gemini)  (Pydantic)")